# Dunnhumby seed 43 — standalone M4-B factorial repeat

M1의 binary graph를 그대로 두고 M5-B와 같은 N/V 분리 행가중치만 적용합니다. 기존 M1·M3·M5-B 300 epoch 결과를 재사용하며 새 학습은 M4-B 한 arm뿐입니다. 개발분할만 사용하고 test·holdout은 열지 않습니다.

In [ ]:
from pathlib import Path
import json, os, subprocess, sys
from google.colab import drive

if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')
try:
    next(Path('/content/drive/MyDrive').iterdir(), None)
except OSError:
    drive.mount('/content/drive', force_remount=True)

SOURCE_COMMIT = '2f13a3c535cdb9583faef0c38c73f604badebf46'
REPO = Path('/content/clv-m4-split-nv-' + SOURCE_COMMIT[:12])
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/jung-un/clv-m2-lightgcn-runner.git', str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin'], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', '--detach', SOURCE_COMMIT], check=True)
assert subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip() == SOURCE_COMMIT
os.chdir(REPO)
sys.path.insert(0, str(REPO))

import torch
assert torch.cuda.is_available(), '학습에는 GPU 런타임을 사용하세요.'
import clv_m5_m3_m4_split_nv_screen as screen
screen.self_test()
cfg = screen.configure(
    seed=43,
    out_dir='/content/drive/MyDrive/논문/data/results_v3_dunnhumby_clv_m4_split_nv_standalone_s43_v1',
)
print(json.dumps({'source_commit': SOURCE_COMMIT, 'seed': 43, 'epochs': cfg.epochs, 'out_dir': cfg.out_dir}, ensure_ascii=False, indent=2))

In [ ]:
result = screen.run_standalone_m4_b(cfg)

In [ ]:
from IPython.display import display

key = [
    'model_id', 'seed', 'epoch',
    'recall@10', 'ndcg@10', 'recall@20', 'ndcg@20', 'recall@50', 'ndcg@50',
    'price_purchase_amount_weighted_hit@10', 'vndcg@10',
    'price_purchase_amount_weighted_hit@20', 'vndcg@20',
    'price_purchase_amount_weighted_hit@50', 'vndcg@50',
]
display(result['absolute'].loc[result['absolute'].epoch.isin([100, 300]), key].sort_values(['epoch', 'model_id']))
display(result['comparison'].loc[result['comparison'].epoch.eq(300)].sort_values(['metric', 'model_id', 'reference']))
print(json.dumps(result['reading'], ensure_ascii=False, indent=2))

In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import files

zip_path = Path('/content/m4_split_nv_standalone_seed43_results.zip')
with ZipFile(zip_path, 'w', ZIP_DEFLATED) as archive:
    for path in result['paths'].values():
        path = Path(path)
        archive.write(path, arcname=path.name)
print('다운로드:', zip_path)
files.download(str(zip_path))